# TripFit — EXAONE LoRA / QLoRA 파인튜닝

부산 호텔·식당·관광지 리뷰를 아래 구조로 추출하도록 `EXAONE-3.5-2.4B-Instruct`를 학습합니다.

```json
{"traveler_context": [...], "aspects": [{"category": "...", "attribute": "...", "sentiment": "...", "evidence": "원문 그대로"}]}
```

💡 **LoRA**는 원본 모델 가중치는 고정하고 작은 어댑터만 학습하는 방식입니다. **QLoRA**는 원본 모델을 4비트로 줄여 LoRA를 학습하므로 VRAM이 적은 GPU에서 더 적합합니다.

## 실행 전 확인

- `datasets/train.jsonl` (Silver 3,087개), `datasets/validation.jsonl` (Silver 371개), `datasets/test.jsonl` (Gold 172개)을 **그대로** 사용합니다.
- `test.jsonl`은 Gold 정답이므로 학습에 절대 넣지 않습니다.
- 기본값은 8GB GPU에 안전한 `qlora`이며, LoRA 비교 시 커널을 재시작한 뒤 `TRAINING_MODE = "lora"`로 바꾸세요.
- `RUN_TRAINING = True`로 바꾸기 전에는 학습이 실행되지 않습니다. `.env`의 HF/W&B 키 값은 출력하지 않습니다.

## 0. 환경 및 실행 설정

프로젝트 루트에서 `uv sync` 후 `travel-planner/.venv` 커널을 사용합니다.
현재 데이터와 RTX 4060 Laptop 8GB를 기준으로 조정한 **권장 시작 설정**입니다.
학습 실험을 하지 않았으므로 최적 성능이 입증된 값은 아닙니다.

| 항목 | 설정 | 근거 |
| --- | --- | --- |
| 최대 입력+정답 길이 | 640 | EXAONE 토큰 실측 train 최대 544, validation 최대 581. 기존 512에서 train 21개/validation 3개 잘림 |
| 학습률 | 1e-4 | Silver 라벨의 잡음을 과하게 학습하지 않도록 2e-4보다 보수적인 시작값 |
| 최대 epoch | 3 | 1회 반복에서 학습을 끝내지 않고 검증 결과로 종료·최고 모델 선택 |
| 실제 batch / 누적 | 1 / 16 | 8GB 메모리를 고려한 유효 batch 16 |
| rank / alpha / dropout | 16 / 32 / 0.05 | 기존 adapter 용량을 유지; 무조건 크게 늘리지 않음 |
| 검증·저장 / 조기 종료 | 50 update마다 / 3회 연속 미개선 | 371개 Validation 전체의 정답 토큰 오차로 선택 |
| warmup | 총 update의 5%를 정수 step으로 계산 | 현재 TRL은 `warmup_ratio` 대신 `warmup_steps` 지원 |

💡 **epoch**는 학습 데이터를 한 번 훑는 단위, **유효 batch**는 한 번 가중치를 갱신할 때 모으는 샘플 수입니다.
**조기 종료**는 검증 오차가 개선되지 않을 때 학습을 멈추는 기능입니다.

설정 선택에는 **Validation만** 사용합니다. Gold Test는 모든 선택이 끝난 뒤 최종 비교에만 사용합니다.
현재 train/validation은 합성 Silver이므로 검증 오차가 낮다고 사람 정답 정확도까지 보장하지는 않습니다.
추후 `5e-5 / 1e-4 / 2e-4`를 같은 Validation에서 비교할 수 있으나, 이 노트북은 자동 탐색을 실행하지 않습니다.

참고: [TRL SFT](https://huggingface.co/docs/trl/sft_trainer),
[EXAONE 공식 모델](https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct).

In [12]:
from pathlib import Path
import os
import random
import json
import gc
import math
import hashlib
from collections import Counter

import torch
from dotenv import load_dotenv


def find_project_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "datasets" / "manifest.json").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트의 datasets/manifest.json을 찾지 못했습니다.")


PROJECT_ROOT = find_project_root()
DATASET_DIR = PROJECT_ROOT / "datasets"
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "exaone_review_extraction"

load_dotenv(PROJECT_ROOT / ".env", override=False)
HF_TOKEN = os.getenv("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"
MODEL_REVISION = "ccce25bd39c141fe053e0bc75818a8f5fe962802"  # 확인한 공식 코드/가중치 버전 고정
TRAINING_MODE = "qlora"  # "qlora" (권장) 또는 "lora". 모드를 바꾸면 커널을 재시작하세요.
RUN_TRAINING = True
USE_WANDB = True
RUN_BASELINE_SAMPLE = False
RUN_GOLD_TEST_SAMPLE = True

# 실험 시작값: 두 모드 비교 시 같은 값으로 유지하세요.
SEED = 42
MAX_LENGTH = 640
MAX_NEW_TOKENS = 384  # train/validation 정답 최대 308토큰 + 여유
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
NUM_TRAIN_EPOCHS = 3  # 조기 종료를 적용하는 상한
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.05
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EVAL_SAVE_STEPS = 50
EARLY_STOPPING_PATIENCE = 3
RUN_TAG = "length640-lr1e4-v2"  # 다른 실험은 이름을 바꿔 기록 분리

assert TRAINING_MODE in {"lora", "qlora"}
OUTPUT_DIR = OUTPUT_ROOT / f"{MODEL_ID.rsplit('/', 1)[-1]}-{TRAINING_MODE}-seed{SEED}-{RUN_TAG}"

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("프로젝트:", PROJECT_ROOT)
print("모드:", TRAINING_MODE)
print("출력 경로:", OUTPUT_DIR)
print("HF_TOKEN 설정:", bool(HF_TOKEN), "| WANDB_API_KEY 설정:", bool(WANDB_API_KEY))

프로젝트: /home/sang/travel-planner
모드: qlora
출력 경로: /home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2
HF_TOKEN 설정: True | WANDB_API_KEY 설정: True


In [2]:
import accelerate
import bitsandbytes
import datasets
import peft
import transformers
import trl

print("Python/PyTorch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("LoRA/QLoRA 학습에는 CUDA NVIDIA GPU가 필요합니다.")

GPU_NAME = torch.cuda.get_device_name(0)
TOTAL_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16_OK = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16

print(f"GPU: {GPU_NAME} ({TOTAL_VRAM_GB:.1f} GB)")
print("학습 dtype:", COMPUTE_DTYPE)
if TRAINING_MODE == "lora" and TOTAL_VRAM_GB < 12:
    print("주의: 12GB 미만 GPU에서는 표준 LoRA가 OOM 날 수 있습니다. 먼저 QLoRA를 권장합니다.")

/home/sang/travel-planner/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python/PyTorch: 2.14.0+cu130
transformers: 5.17.0
peft: 0.21.0
trl: 1.14.0
bitsandbytes: 0.50.2
CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU (8.0 GB)
학습 dtype: torch.bfloat16


## 1. 통합 데이터셋 검증 및 불러오기

`manifest.json`에 기록된 분할을 다시 만들지 않습니다. 이 셀은 파일 수, tier, split, train/test ID 겹침, Evidence 원문 포함 여부를 검증한 뒤 `Dataset`으로 만듭니다.

In [3]:
TRAIN_PATH = DATASET_DIR / "train.jsonl"
VALIDATION_PATH = DATASET_DIR / "validation.jsonl"
TEST_PATH = DATASET_DIR / "test.jsonl"
MANIFEST_PATH = DATASET_DIR / "manifest.json"


def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def validate_records(records: list[dict], expected_tier: str, expected_split: str) -> list[str]:
    errors = []
    required = {"review_id", "category", "review", "label", "tier", "split"}
    for index, record in enumerate(records, start=1):
        missing = required - record.keys()
        if missing:
            errors.append(f"{index}: 필수 필드 누락 {sorted(missing)}")
            continue
        if record["tier"] != expected_tier:
            errors.append(f"{record['review_id']}: tier={record['tier']!r}")
        if record["split"] != expected_split:
            errors.append(f"{record['review_id']}: split={record['split']!r}")
        label = record["label"]
        if set(label) != {"traveler_context", "aspects"}:
            errors.append(f"{record['review_id']}: label 키가 스키마와 다릅니다.")
            continue
        for aspect in label["aspects"]:
            if set(aspect) != {"category", "attribute", "sentiment", "evidence"}:
                errors.append(f"{record['review_id']}: aspect 키가 스키마와 다릅니다.")
                continue
            if aspect["evidence"] not in record["review"]:
                errors.append(f"{record['review_id']}: evidence가 review 원문에 없습니다.")
    return errors


train_records = read_jsonl(TRAIN_PATH)
validation_records = read_jsonl(VALIDATION_PATH)
test_records = read_jsonl(TEST_PATH)

errors = []
errors += validate_records(train_records, "silver", "train")
errors += validate_records(validation_records, "silver", "val")
errors += validate_records(test_records, "gold", "test")

train_ids = {row["review_id"] for row in train_records}
validation_ids = {row["review_id"] for row in validation_records}
test_ids = {row["review_id"] for row in test_records}
overlap = (train_ids & validation_ids) | (train_ids & test_ids) | (validation_ids & test_ids)
if overlap:
    errors.append(f"분할 간 review_id 중복: {sorted(overlap)[:5]}")
if errors:
    raise ValueError("데이터셋 검증 실패:\n" + "\n".join(errors[:20]))

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print("데이터셋:", manifest["dataset_name"], "| 전략:", manifest["strategy"])
for split_name, rows in (("train", train_records), ("validation", validation_records), ("test (Gold)", test_records)):
    print(f"{split_name:16s} {len(rows):4d}개 | category={dict(sorted(Counter(x['category'] for x in rows).items()))}")
print("검증 통과: test Gold는 train/validation과 review_id가 겹치지 않습니다.")

데이터셋: busan_review_sft_v1 | 전략: A_review_only
train            3087개 | category={'attraction': 1186, 'hotel': 651, 'restaurant': 1250}
validation        371개 | category={'attraction': 154, 'hotel': 80, 'restaurant': 137}
test (Gold)       172개 | category={'attraction': 54, 'hotel': 53, 'restaurant': 65}
검증 통과: test Gold는 train/validation과 review_id가 겹치지 않습니다.


## 2. EXAONE 토크나이저 로드

공식 EXAONE 전용 코드가 필요하여 `trust_remote_code=True`를 사용하고 확인한 revision으로 고정합니다.
토크나이저로 전처리·길이 검증을 먼저 끝낸 다음 모델 가중치를 로드합니다.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN, trust_remote_code=True,
)
tokenizer.clean_up_tokenization_spaces = False  # evidence의 원문 공백 보존
tokenizer.padding_side = "right"
if tokenizer.eos_token is None:
    raise ValueError("EXAONE tokenizer의 eos_token을 찾지 못했습니다.")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("EOS:", repr(tokenizer.eos_token), "| PAD:", repr(tokenizer.pad_token))

EOS: '[|endofturn|]' | PAD: '[PAD]'


## 3. 정답 경계와 토큰 길이 검증

EXAONE은 `[|assistant|]` 뒤에 JSON을 붙여 한 번에 토큰화하면 경계 토큰이 달라집니다.
입력과 정답을 각각 토큰화한 뒤 이어 붙이고, 입력 labels를 `-100`으로 지정합니다.
정답 첫 `{`부터 종료 토큰까지 전부 학습하며, 640토큰을 넘는 데이터는 잘라 학습하지 않고 오류로 알립니다.
길이 측정과 하이퍼파라미터 선택에 Gold Test 정답은 사용하지 않습니다.

In [5]:
from datasets import Dataset

SYSTEM_PROMPT = """당신은 부산 장소 리뷰 정보 추출기다.
주어진 카테고리와 리뷰 원문에서만 정보를 추출하라.
반드시 JSON 객체 하나만 출력하라. 정확한 형식은
{\"traveler_context\": [...], \"aspects\": [{\"category\": \"...\", \"attribute\": \"...\", \"sentiment\": \"...\", \"evidence\": \"원문 그대로의 연속 구절\"}]} 이다.
리뷰에 없는 사실을 만들지 말고, evidence는 반드시 리뷰 원문을 그대로 인용하라."""


def review_user_message(record: dict) -> str:
    return f"[카테고리]\n{record['category']}\n\n[리뷰]\n{record['review']}"


def format_for_sft(record: dict) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": review_user_message(record)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    completion = json.dumps(record["label"], ensure_ascii=False, separators=(",", ":")) + tokenizer.eos_token
    return {"prompt": prompt, "completion": completion}


def tokenize_for_sft(record: dict) -> dict:
    formatted = format_for_sft(record)
    prompt_ids = tokenizer(formatted["prompt"], add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(formatted["completion"], add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + completion_ids
    if len(input_ids) > MAX_LENGTH:
        raise ValueError(
            f"{record['review_id']}: {len(input_ids)} > MAX_LENGTH={MAX_LENGTH}. "
            "정답을 자르지 않습니다. 길이 설정을 확인하세요."
        )
    if not completion_ids or completion_ids[-1] != tokenizer.eos_token_id:
        raise ValueError(f"{record['review_id']}: 정답 종료 토큰 누락")
    decoded = tokenizer.decode(completion_ids[:-1], clean_up_tokenization_spaces=False)
    if json.loads(decoded) != record["label"]:
        raise ValueError(f"{record['review_id']}: 정답 토큰화 왕복 검증 실패")
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + completion_ids,
    }


train_dataset = Dataset.from_list([tokenize_for_sft(row) for row in train_records])
validation_dataset = Dataset.from_list([tokenize_for_sft(row) for row in validation_records])
length_report = {}
for name, dataset in (("train", train_dataset), ("validation", validation_dataset)):
    lengths = [len(row["input_ids"]) for row in dataset]
    completions = [sum(t != -100 for t in row["labels"]) for row in dataset]
    length_report[name] = {
        "count": len(dataset), "max_tokens": max(lengths),
        "mean_tokens": round(sum(lengths) / len(lengths), 2),
        "max_completion_tokens": max(completions),
        "over_old_512": sum(n > 512 for n in lengths),
        "over_current_limit": sum(n > MAX_LENGTH for n in lengths),
    }
    print(name, length_report[name])

# 단일 GPU, batch=1, 누적=16: ceil(3087/16)=193 updates/epoch.
EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
UPDATES_PER_EPOCH = math.ceil(len(train_dataset) / EFFECTIVE_BATCH_SIZE)
PLANNED_UPDATES = math.ceil(UPDATES_PER_EPOCH * NUM_TRAIN_EPOCHS)
WARMUP_STEPS = math.ceil(PLANNED_UPDATES * WARMUP_RATIO)
print(f"유효 batch={EFFECTIVE_BATCH_SIZE}, 최대 updates={PLANNED_UPDATES}, warmup={WARMUP_STEPS}")

train {'count': 3087, 'max_tokens': 544, 'mean_tokens': 356.39, 'max_completion_tokens': 308, 'over_old_512': 21, 'over_current_limit': 0}
validation {'count': 371, 'max_tokens': 581, 'mean_tokens': 361.35, 'max_completion_tokens': 292, 'over_old_512': 3, 'over_current_limit': 0}
유효 batch=16, 최대 updates=579, warmup=29


In [6]:
# 길이 검증을 통과한 다음 실행합니다. 모드 변경 시 커널을 재시작하세요.
model_kwargs = {
    "token": HF_TOKEN,
    "revision": MODEL_REVISION,
    "trust_remote_code": True,
    "dtype": COMPUTE_DTYPE,
    "device_map": {"": 0},
    "attn_implementation": "sdpa",
}
if TRAINING_MODE == "qlora":
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True,
    )
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

# EXAONE 3.5 공식 원격 코드는 create_causal_mask의 이전 인자명
# input_embeds/cache_position을 사용합니다. Transformers 5.17 API로 변환합니다.
import importlib
from transformers.masking_utils import create_causal_mask as _hf_create_causal_mask

_exaone_module = importlib.import_module(model.__class__.__module__)
if not getattr(_exaone_module, "_tripfit_mask_compat", False):
    def _tripfit_create_causal_mask(
        *, config, input_embeds=None, inputs_embeds=None, attention_mask=None,
        past_key_values=None, position_ids=None, **_ignored,
    ):
        return _hf_create_causal_mask(
            config=config,
            inputs_embeds=inputs_embeds if inputs_embeds is not None else input_embeds,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            position_ids=position_ids,
        )
    _exaone_module.create_causal_mask = _tripfit_create_causal_mask
    _exaone_module._tripfit_mask_compat = True

model.config.use_cache = False
# EXAONE 3.5의 전용 코드가 최신 PEFT가 찾는 입력 임베딩 이름을 선언하지 않습니다.
# wte를 표준 입력 임베딩으로 연결해야 tied embedding 검사가 통과합니다.
if not hasattr(model, "transformer") or not hasattr(model.transformer, "wte"):
    raise RuntimeError("EXAONE 입력 임베딩(wte)을 찾지 못했습니다. MODEL_REVISION을 확인하세요.")
model.transformer._input_embed_layer = "wte"
print(f"모델 로드 완료: {MODEL_ID} / {TRAINING_MODE}")
print(f"현재 할당 VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/sang/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/sang/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.


Loading weights: 100%|██████████| 272/272 [00:05<00:00, 46.46it/s]


모델 로드 완료: LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct / qlora
현재 할당 VRAM: 1.54 GB


## 4. LoRA 어댑터 설정

`all-linear`은 EXAONE 내부의 선형 계층에 LoRA를 붙여 아키텍처별 모듈 이름을 하드코딩하지 않는 설정입니다. 학습 가능한 adapter 파라미터가 0개면 즉시 에러를 내도록 점검합니다.

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def clear_cuda_cache() -> None:
    gc.collect()
    torch.cuda.empty_cache()


def ensure_lora_trainable(peft_model):
    if not hasattr(peft_model, "peft_config"):
        raise RuntimeError("LoRA adapter가 연결되지 않았습니다.")
    if hasattr(peft_model, "set_adapter"):
        peft_model.set_adapter("default")
    for name, parameter in peft_model.named_parameters():
        if "lora_" in name.lower():
            parameter.requires_grad_(True)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    lora_trainable = sum(p.numel() for n, p in peft_model.named_parameters() if "lora_" in n.lower() and p.requires_grad)
    if lora_trainable == 0:
        raise RuntimeError("Trainable LoRA parameter가 0개입니다. 모델 로드 셀부터 다시 실행하세요.")
    print(f"trainable params: {trainable:,} | trainable LoRA params: {lora_trainable:,}")
    return peft_model


if hasattr(model, "peft_config"):
    raise RuntimeError("이미 adapter가 붙어 있습니다. 커널을 재시작하세요.")

if TRAINING_MODE == "qlora":
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
else:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.config.use_cache = False
model = ensure_lora_trainable(model)

trainable params: 21,995,520 | trainable LoRA params: 21,995,520


## 5. Trainer 생성 및 loss masking 확인

💡 **Completion-only loss**는 리뷰 입력문이 아니라 정답 JSON 토큰에만 오차를 계산하는 설정입니다. 따라서 모델은 리뷰를 그대로 따라 쓰는 대신 구조화된 결과를 생성하도록 학습합니다.

In [8]:
from trl import SFTConfig, SFTTrainer
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling
from transformers import EarlyStoppingCallback, TrainerCallback


class EvaluateLastStepCallback(TrainerCallback):
    # 마지막 update가 50의 배수가 아니어도 최종 상태를 검증/저장합니다.
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step >= state.max_steps:
            control.should_evaluate = True
            control.should_save = True
        return control


training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    weight_decay=0.0,
    optim="paged_adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,
    save_strategy="steps",
    save_steps=EVAL_SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    prediction_loss_only=True,
    bf16=BF16_OK,
    fp16=not BF16_OK,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    # 현재 TRL의 메모리 절약 cross-entropy. 모델의 output head에 LoRA는 붙이지 않습니다.
    loss_type="chunked_nll",
    dataset_kwargs={"skip_prepare_dataset": True},  # 위에서 만든 labels 그대로 사용
    packing=False,
    padding_free=False,
    eos_token=tokenizer.eos_token,
    pad_token=tokenizer.pad_token,
    report_to="wandb" if USE_WANDB else "none",
    run_name=f"tripfit-{TRAINING_MODE}-{RUN_TAG}-seed{SEED}",
    seed=SEED,
    data_seed=SEED,
)
data_collator = DataCollatorForLanguageModeling(
    pad_token_id=tokenizer.pad_token_id, pad_to_multiple_of=8,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        EvaluateLastStepCallback(),
    ],
)
trainer.model = ensure_lora_trainable(trainer.model)

# 가장 긴 학습 샘플도 정답 전체가 남아 있는지 확인합니다.
longest_index = max(range(len(train_dataset)), key=lambda i: len(train_dataset[i]["input_ids"]))
example = train_dataset[longest_index]
diagnostic_batch = data_collator([example])
labels = diagnostic_batch["labels"][0]
expected = torch.tensor(example["labels"])
assert torch.equal(labels[:len(expected)], expected)
assert (labels[len(expected):] == -100).all()
answer_ids = labels[labels != -100].tolist()
assert answer_ids[-1] == tokenizer.eos_token_id
assert json.loads(tokenizer.decode(answer_ids[:-1], clean_up_tokenization_spaces=False)) == train_records[longest_index]["label"]
print("검증 통과: 입력/padding은 -100, JSON 첫 글자부터 EOS까지 정답 전체에 loss 적용")
del diagnostic_batch

trainable params: 21,995,520 | trainable LoRA params: 21,995,520
검증 통과: 입력/padding은 -100, JSON 첫 글자부터 EOS까지 정답 전체에 loss 적용


/tmp/ipykernel_338619/3332732758.py:15: FutureWarning: `pad_token` is deprecated and will be removed in v2.0.0. Set `tokenizer.pad_token` directly and pass it as `processing_class` to the trainer instead.
  training_args = SFTConfig(


## 6. 학습 실행 및 어댑터 저장

`RUN_TRAINING = True`일 때만 실행됩니다. QLoRA도 전체 EXAONE 가중치가 아니라 작은 LoRA 어댑터만 저장합니다.

In [9]:
if not RUN_TRAINING:
    print("학습 미실행: 0번 설정 셀에서 RUN_TRAINING=True로 바꾼 뒤 이 셀을 다시 실행하세요.")
else:
    if USE_WANDB:
        import wandb
        if not WANDB_API_KEY:
            raise RuntimeError("USE_WANDB=True인데 WANDB_API_KEY가 없습니다.")
        os.environ.setdefault("WANDB_PROJECT", "tripfit-review-extraction")
        wandb.login(key=WANDB_API_KEY, relogin=True)

    clear_cuda_cache()
    torch.cuda.reset_peak_memory_stats()
    trainer.model = ensure_lora_trainable(trainer.model)
    train_result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR / "adapter"))
    tokenizer.save_pretrained(OUTPUT_DIR / "adapter")

    peak_vram = torch.cuda.max_memory_allocated() / 1024**3
    print(f"학습 완료: {train_result.metrics.get('train_runtime', float('nan')):.1f}초")
    print(f"최대 할당 VRAM: {peak_vram:.2f} GB")
    print("저장 위치:", OUTPUT_DIR / "adapter")
    print("선택된 checkpoint:", trainer.state.best_model_checkpoint)
    print("최저 validation loss:", trainer.state.best_metric)
    run_summary = {
        "model_id": MODEL_ID, "revision": MODEL_REVISION, "mode": TRAINING_MODE,
        "seed": SEED, "max_length": MAX_LENGTH, "learning_rate": LEARNING_RATE,
        "max_epochs": NUM_TRAIN_EPOCHS, "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "warmup_steps": WARMUP_STEPS, "planned_updates": PLANNED_UPDATES,
        "actual_updates": trainer.state.global_step,
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_loss": trainer.state.best_metric,
        "peak_vram_gb": peak_vram, "metrics": train_result.metrics,
        "length_report": length_report,
        "dataset_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                           for p in (TRAIN_PATH, VALIDATION_PATH, TEST_PATH)},
    }
    (OUTPUT_DIR / "run_summary.json").write_text(
        json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    if USE_WANDB:
        wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/sang/.netrc
wandb: Currently logged in as: hinoonyaso (hinoonyaso-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


trainable params: 21,995,520 | trainable LoRA params: 21,995,520


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.080591,0.070267,0.089179,285213.000000,0.977077
100,0.045877,0.054310,0.062524,571518.000000,0.981500
150,0.046321,0.051899,0.060497,856271.000000,0.982229
200,0.031624,0.048647,0.051815,1139618.000000,0.982854
250,0.033471,0.047522,0.045480,1424063.000000,0.984000
300,0.032551,0.044258,0.043529,1706108.000000,0.984639
350,0.031513,0.044076,0.043071,1994197.000000,0.984457
400,0.025691,0.043274,0.038814,2280643.000000,0.985403
450,0.027303,0.043736,0.035782,2564688.000000,0.985200
500,0.022515,0.043457,0.035372,2854270.000000,0.985201


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


학습 완료: 5624.6초
최대 할당 VRAM: 3.20 GB
저장 위치: /home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/adapter
선택된 checkpoint: /home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/checkpoint-400
최저 validation loss: 0.04327420890331268


eval/entropy,█▅▄▃▂▂▂▁▁▁▁
eval/loss,█▄▃▂▂▁▁▁▁▁▁
eval/mean_token_accuracy,▁▅▅▆▇▇▇████
eval/num_tokens,▁▂▂▃▄▄▅▆▇▇█
eval/runtime,▄▇█▁██▇▇▇▂▇
eval/samples_per_second,▅▂▁█▁▁▂▁▂▇▂
eval/steps_per_second,▅▂▁█▁▁▂▁▂▇▂
train/entropy,█▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▁▁▁▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇████
+5,...


## 7. Gold Test 샘플 생성

Gold Test 172개는 최종 비교용입니다. 이 셀은 정답을 프롬프트에 넣지 않고 모델 출력을 생성합니다. 전체 172개 자동 채점 기준은 팀에서 확정한 뒤 확장하세요.

In [13]:
def generate_extraction(record: dict, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": review_user_message(record)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(next(model.parameters()).device)
    model.eval()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    answer_ids = generated[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()


if not RUN_GOLD_TEST_SAMPLE:
    print("Gold Test 생성 미실행: RUN_GOLD_TEST_SAMPLE=True로 바꾸면 첫 Gold 샘플만 생성합니다.")
else:
    gold_example = test_records[0]
    prediction = generate_extraction(gold_example)
    print("[리뷰]", gold_example["review"])
    print("\n[모델 출력]", prediction)
    print("\n[Gold 정답 — 비교용, 모델 입력에는 사용하지 않음]", json.dumps(gold_example["label"], ensure_ascii=False))

[transformers] The remote code model you are currently using seems to expect `cache_position`. This arg has been removed from the Transformers library, and will stop being created in `generate` even for remote code models in a future release. Please open a PR on the remote code hub repo to remove any usage of `cache_position`.


[리뷰] 남자친구랑 커플 여행으로 나인호텔에 묵었는데, 호텔 시설 같은 건 전반적으로 만족스러웠어요. 다만 주차 공간이 넉넉하지 않아서 차 가져가신 분들은 조금 불편하실 것 같네요. 덕분에 해운대 바닷가 근처에서 편하게 휴식하며 좋은 시간 보낼 수 있어서 좋았습니다.

[모델 출력] {"traveler_context":["couple"],"aspects":[{"category":"room_condition","attribute":"well_kept","sentiment":"positive","evidence":"호텔 시설 같은 건 전반적으로 만족스러웠어요."},{"category":"parking_availability","attribute":"limited","sentiment":"negative","evidence":"주차 공간이 넉넉하지 않아서"},{"category":"view_quality","attribute":"good","sentiment":"positive","evidence":"해운대 바닷가 근처에서 편하게 휴식하며 좋은 시간 보낼 수 있어서 좋았습니다."}]}

[Gold 정답 — 비교용, 모델 입력에는 사용하지 않음] {"traveler_context": ["couple"], "aspects": [{"category": "room_condition", "attribute": "well_kept", "sentiment": "positive", "evidence": "호텔 시설 같은 건 전반적으로 만족스러웠어요."}, {"category": "parking_availability", "attribute": "limited", "sentiment": "negative", "evidence": "주차 공간이 넉넉하지 않아서"}, {"category": "view_quality", "attribute": "good", "sentiment": "positive", "evidence": "해운대 바닷가 근처에서 편하게 휴식하며 좋은 시간 보낼 수 있어

## 실행 체크리스트

1. `travel-planner/.venv` 커널을 선택하고 설정·데이터·토큰 길이 셀부터 확인합니다.
2. `RUN_TRAINING=True`로 실행하면 최대 3 epoch 내에서 Validation 기준 조기 종료 및 최고 checkpoint 복원이 적용됩니다.
3. 학습이 끝나면 adapter와 `run_summary.json`에 실제 update 수·최저 검증 오차·설정·데이터 해시가 저장됩니다.
4. LoRA 비교는 커널을 재시작하고 `TRAINING_MODE="lora"`로 바꿉니다. 동일 데이터·설정으로 비교하되 실제 종료 update 수도 함께 보고합니다.
5. Gold Test는 설정 확정 후 최종 평가에만 사용합니다. 결과를 보고 다시 튜닝하면 독립 테스트가 아니게 됩니다.
6. 8GB에서 메모리가 부족하면 QLoRA와 batch=1을 유지하고 다른 GPU 작업을 종료한 뒤 확인하세요. 누적 횟수만 늘려도 단일 batch의 메모리는 줄지 않습니다. 길이를 512로 되돌리면 일부 JSON 정답이 잘립니다.

## 8. 학습 어댑터 병합 → GGUF 변환 → Ollama 등록

💡 **병합(merge)**은 학습한 LoRA 어댑터의 변경분을 원본 EXAONE 가중치에 합치는 과정입니다. Ollama는 이처럼 완성된 모델을 GGUF 형식으로 받아 추론합니다.

이 단계는 학습과 별도입니다. CPU RAM과 디스크 여유 공간이 필요하며, `RUN_EXPORT_TO_OLLAMA=True`일 때만 실행됩니다. 기존 `exaone3.5:2.4b`를 덮어쓰지 않고 `exaone-tripfit-{TRAINING_MODE}:{RUN_TAG}`라는 새 모델을 만듭니다.

In [15]:
import shutil
import subprocess
import sys

RUN_EXPORT_TO_OLLAMA = True
RUN_OLLAMA_SMOKE_TEST = False

# 현재 워크스페이스에 있는 llama.cpp 경로입니다. 다른 컴퓨터에서는 이 값만 바꾸세요.
LLAMA_CPP_DIR = Path("/home/sang/qlora_ft_ex/llama.cpp")
CONVERTER = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
QUANTIZER = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"

ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_DIR = OUTPUT_DIR / "merged_safetensors"
GGUF_DIR = OUTPUT_DIR / "gguf"
GGUF_F16 = GGUF_DIR / "exaone-tripfit-lora-f16.gguf"
GGUF_Q4_K_M = GGUF_DIR / "exaone-tripfit-lora-q4_k_m.gguf"
MODELFILE_PATH = OUTPUT_DIR / "Modelfile"
OLLAMA_MODEL_NAME = f"exaone-tripfit-{TRAINING_MODE}:{RUN_TAG}"

if not RUN_EXPORT_TO_OLLAMA:
    print("Ollama 내보내기 미실행: 학습 완료 후 RUN_EXPORT_TO_OLLAMA=True로 바꾸세요.")
else:
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f"저장된 adapter가 없습니다: {ADAPTER_DIR}")
    if not CONVERTER.exists() or not QUANTIZER.exists():
        raise FileNotFoundError("llama.cpp 변환기 또는 llama-quantize를 찾지 못했습니다. LLAMA_CPP_DIR을 확인하세요.")
    if shutil.which("ollama") is None:
        raise RuntimeError("ollama 명령을 찾지 못했습니다.")

    # 학습에 쓴 4bit 모델이 아니라 원본 BF16/FP16 EXAONE을 CPU에 읽어 adapter를 안전하게 병합합니다.
    # 병합 중에는 GPU 모델을 제거해 VRAM을 비웁니다.
    if "trainer" in globals():
        del trainer
    if "model" in globals():
        del model
    clear_cuda_cache()

    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        dtype=torch.float16,
        revision=MODEL_REVISION, trust_remote_code=True,
        device_map={"": "cpu"},
        low_cpu_mem_usage=True,
    )
    if not hasattr(base_model, "transformer") or not hasattr(base_model.transformer, "wte"):
        raise RuntimeError("EXAONE 입력 임베딩(wte)을 찾지 못했습니다. MODEL_REVISION을 확인하세요.")
    base_model.transformer._input_embed_layer = "wte"
    merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).merge_and_unload(safe_merge=True)
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(MERGED_DIR)
    del merged_model, base_model
    gc.collect()

        # EXAONE 3.5의 RMSNorm 메타데이터 키 호환 패치
    EXAONE_CONVERTER_SOURCE = LLAMA_CPP_DIR / "conversion" / "exaone.py"
    PATCH_LINE = 'self.gguf_writer.add_layer_norm_rms_eps(hparams["layer_norm_epsilon"])'

    converter_source = EXAONE_CONVERTER_SOURCE.read_text(encoding="utf-8")
    if PATCH_LINE not in converter_source:
        anchor = (
            "        self.gguf_writer.add_rope_dimension_count("
            'int(rotary_factor * (hparams["hidden_size"] // hparams["num_attention_heads"])))'
        )
        if anchor not in converter_source:
            raise RuntimeError("EXAONE 변환기 패치 위치를 찾지 못했습니다.")

        converter_source = converter_source.replace(
            anchor,
            f"{anchor}\n        {PATCH_LINE}",
            1,
        )
        EXAONE_CONVERTER_SOURCE.write_text(converter_source, encoding="utf-8")
        print("EXAONE GGUF RMSNorm 호환 패치 적용 완료")
        
    GGUF_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [sys.executable, str(CONVERTER), str(MERGED_DIR), "--outfile", str(GGUF_F16), "--outtype", "f16"],
        check=True,
    )
    subprocess.run([str(QUANTIZER), str(GGUF_F16), str(GGUF_Q4_K_M), "Q4_K_M"], check=True)

    # 현재 설치된 exaone3.5:2.4b의 대화 형식을 유지합니다.
    modelfile = f"""FROM {GGUF_Q4_K_M}
TEMPLATE \"\"\"{{{{- range $i, $_ := .Messages }}}}
{{{{- $last := eq (len (slice $.Messages $i)) 1 -}}}}
{{{{ if eq .Role "system" }}}}[|system|]{{{{ .Content }}}}[|endofturn|]
{{{{ continue }}}}
{{{{ else if eq .Role "user" }}}}[|user|]{{{{ .Content }}}}
{{{{ else if eq .Role "assistant" }}}}[|assistant|]{{{{ .Content }}}}[|endofturn|]
{{{{ end }}}}
{{{{- if and (ne .Role "assistant") $last }}}}[|assistant|]{{{{ end }}}}
{{{{- end -}}}}\"\"\"
SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
PARAMETER temperature 0
PARAMETER num_ctx 2048
PARAMETER num_predict {MAX_NEW_TOKENS}
PARAMETER stop [|endofturn|]
"""
    MODELFILE_PATH.write_text(modelfile, encoding="utf-8")
    subprocess.run(["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(MODELFILE_PATH)], check=True)
    print("Ollama 모델 등록 완료:", OLLAMA_MODEL_NAME)
    print("GGUF:", GGUF_Q4_K_M)

    if RUN_OLLAMA_SMOKE_TEST:
        sample = test_records[0]
        result = subprocess.run(
            ["ollama", "run", OLLAMA_MODEL_NAME, review_user_message(sample)],
            check=True,
            text=True,
            capture_output=True,
        )
        print(result.stdout)

Writing model shards: 100%|██████████| 3/3 [00:02<00:00,  1.24it/s]


EXAONE GGUF RMSNorm 호환 패치 적용 완료


INFO:hf-to-gguf:Loading model: merged_safetensors
 You can inspect the repository content at https://hf.co//home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/merged_safetensors.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.
INFO:hf-to-gguf:Model architecture: ExaoneForCausalLM
 You can inspect the repository content at https://hf.co//home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/merged_safetensors.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00003.safetensors'
INFO:gguf.gguf_writer:gguf: 

The repository /home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/merged_safetensors contains custom code which must be executed to correctly load the model. You can inspect the repository content at /home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/merged_safetensors .
 You can inspect the repository content at https://hf.co//home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/merged_safetensors.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] 

[transformers] You are using a model of type `exaone` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
INFO:gguf.vocab:Adding 101782 merge(s).
INFO:gguf.vocab:Setting special token type bos to 1
INFO:gguf.vocab:Setting special token type eos to 361
INFO:gguf.vocab:Setting special token type unk to 3
INFO:gguf.vocab:Setting special token type pad to 0
INFO:gguf.vocab:Setting chat_template to {% for message in messages %}{% if loop.first and message['role'] != 'system' %}{{ '[|system|][|endofturn|]
' }}{% endif %}{{ '[|' + message['role'] + '|]' + message['content'] }}{% if message['role'] == 'user' %}{{ '
' }}{% else %}{{ '[|endofturn|]
' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '[|assistant|]' }


llama_quantize: quantize time = 24145.17 ms
llama_quantize:    total time = 24145.17 ms


gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠏ gathering model components ⠏ gathering model components ⠋ gathering model components ⠹ gathering model components ⠹ gathering model components ⠸ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠇ gathering model components ⠋ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components 
copying file sha256:ff8961ccaae01bfcbd88c88d8ae8e716fc461d0beb9cea7e0140cb68ecf6cd19 3% ⠋ gathering model components 
copying file sha256:ff8961ccaae01bfcbd88c88d8ae8e716fc461d0beb9cea7e0140cb68ecf6cd19 8% ⠙ gathering model components 
copying file sha256:ff8961ccaae01bfcbd88c88d8ae8e716fc461d0beb9cea7e0

Ollama 모델 등록 완료: exaone-tripfit-qlora:length640-lr1e4-v2
GGUF: /home/sang/travel-planner/artifacts/exaone_review_extraction/EXAONE-3.5-2.4B-Instruct-qlora-seed42-length640-lr1e4-v2/gguf/exaone-tripfit-lora-q4_k_m.gguf


gathering model components 
copying file sha256:ff8961ccaae01bfcbd88c88d8ae8e716fc461d0beb9cea7e0140cb68ecf6cd19 100% 
parsing GGUF 
verifying conversion 
using existing layer sha256:ff8961ccaae01bfcbd88c88d8ae8e716fc461d0beb9cea7e0140cb68ecf6cd19 
using existing layer sha256:37cddd3bd818d29cacc4a39d1f0c1dd8d4b152c7f0b7277107a6344a4f9e2244 
creating new layer sha256:7e50d947518ddcce9f08ae41b174f8c84043cae9b25ffc7d61348d9113445163 
creating new layer sha256:a6d2910ff2fc2b9ad7112ad370984836990ea7ec709e3b7ba8f1dfc7f0fcbc10 
writing manifest 
success 
